# MI Event Flow Agent 실행 데모

이 노트북은 `skmip-mi-agent` 프로젝트의 `event_flow_agent`가 `/invoke` API를 통해 어떻게 동작하는지 단계별로 확인하기 위한 데모 자료입니다.

전체 흐름은 다음과 같습니다.

1. 원천 record 조회
2. record별 LLM 분석
3. event 단위 추출
4. `tmm_mieventm` 테이블 저장
5. 저장된 event 재조회
6. 최종 요약 생성

> 사전 조건: `skmip_mcp_server`, `skmip-mi-agent`, 로컬 MySQL Docker DB가 실행 중이어야 합니다. rkt


## 1. 실행 환경 확인

아래 값은 현재 데모 환경 기준입니다.

- Agent API: `http://localhost:3002/invoke`
- DB Host: `127.0.0.1`
- DB Port: `13306`
- DB Name: `midb`
- Target Table: `tmm_mieventm`


In [ ]:
AGENT_INVOKE_URL = "http://localhost:3002/invoke"

DB_HOST = "127.0.0.1"
DB_PORT = 13306
DB_NAME = "midb"
DB_USER = "root"

# 보안상 비밀번호는 노트북에 직접 저장하지 않습니다.
# 실행 시 아래 변수에 로컬 DB 비밀번호를 입력하세요.
DB_PASSWORD = ""  # env.txt의 METADATA_DB_PASSWORD의 값을 넣으세요

print("Agent URL:", AGENT_INVOKE_URL)
print("DB:", f"{DB_HOST}:{DB_PORT}/{DB_NAME}")


## 2. API 요청 Body 준비

`limit`은 원천 record 조회 건수입니다. 현재 데모에서는 5건을 기준으로 실행합니다.


In [ ]:
request_body = {
    "input": {
        "limit": 5
    },
    "config": {},
    "kwargs": {}
}

request_body


## 3. 실행 전 DB 상태 확인

API 실행 전 `tmm_mieventm`에 몇 건이 적재되어 있는지 확인합니다.

> 이 단계는 DB 접속 비밀번호가 필요합니다. 비밀번호를 입력하지 않으면 건너뛰어도 됩니다.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

def create_db_engine():
    if not DB_PASSWORD:
        raise ValueError("DB_PASSWORD 값을 먼저 입력하세요.")
    return create_engine(
        f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
        pool_pre_ping=True,
    )

def get_event_count():
    engine = create_db_engine()
    with engine.connect() as conn:
        return conn.execute(text("select count(*) from tmm_mieventm")).scalar_one()

try:
    before_count = get_event_count()
    print("실행 전 tmm_mieventm 건수:", before_count)
except Exception as exc:
    before_count = None
    print("DB 사전 확인을 건너뜁니다:", exc)


## 4. `event_flow_agent` invoke API 실행

아래 셀은 실제 `/invoke` API를 호출합니다. 실행 중 LLM 호출이 포함되어 있어 시간이 걸릴 수 있습니다.


In [ ]:
import requests
import json

AUTH_TOKEN = ""  # env.txt의 API_KEY 값을 넣으세요

headers = {
    "authorization": AUTH_TOKEN,
    "Content-Type": "application/json",
    "aip-user": "test-user"
}

response = requests.post(
    AGENT_INVOKE_URL,
    json=request_body,
    headers=headers,
    timeout=300,
)

print("HTTP Status:", response.status_code)
print(response.text[:1000])

response.raise_for_status()
result = response.json()
result.keys()

## 5. 최종 요약 결과 확인

LLM이 DB에 저장된 event 목록을 바탕으로 생성한 최종 요약입니다.


In [ ]:
output = result["output"]

print(output.get("summary", ""))


## 6. 실행된 노드 확인

`event_flow_agent`가 어떤 순서로 실행되었는지 확인합니다.


In [ ]:
executed_nodes = output.get("executed_nodes", [])
pd.DataFrame({"step": range(1, len(executed_nodes) + 1), "node": executed_nodes})


## 7. 단계별 처리 건수 확인

원천 record 조회, record 분석, event 추출, DB 저장 결과를 요약해서 봅니다.


In [ ]:
summary_metrics = {
    "source_records_count": output.get("source_records_count"),
    "record_analyses_count": output.get("record_analyses_count"),
    "extracted_events_count": output.get("extracted_events_count"),
    "saved_event_ids": output.get("saved_event_ids"),
    "errors": output.get("errors"),
}

summary_metrics


## 8. 저장 후 재조회된 event 목록

`read_rdb` 노드에서 DB를 다시 조회한 결과입니다. 이 값이 최종 요약 LLM의 입력으로 사용됩니다.


In [ ]:
events_df = pd.DataFrame(output.get("rdb_result", []))

display_columns = [
    "mi_event_id",
    "first_created",
    "event_title_ko",
    "one_line_summary",
    "level_1_code",
    "level_2_code",
    "occurred_at",
]

existing_columns = [col for col in display_columns if col in events_df.columns]
events_df[existing_columns] if not events_df.empty else events_df


## 9. 실행 후 DB 상태 확인

API 실행 후 `tmm_mieventm` 테이블에 row가 증가했는지 확인합니다.


In [ ]:
try:
    after_count = get_event_count()
    print("실행 전 건수:", before_count)
    print("실행 후 건수:", after_count)
    if before_count is not None:
        print("증가 건수:", after_count - before_count)
except Exception as exc:
    after_count = None
    print("DB 사후 확인을 건너뜁니다:", exc)


## 10. DB에 적재된 최신 event 상세 확인

실제 `tmm_mieventm` 테이블에 저장된 최신 event를 직접 조회합니다.


In [ ]:
def fetch_latest_events(limit: int = 10) -> pd.DataFrame:
    engine = create_db_engine()
    sql = text("""
        select
            mi_event_id,
            first_created,
            event_title_ko,
            one_line_summary,
            detail_analysis,
            market_and_institution_view,
            occurred_at,
            language,
            dedup_state,
            level_1_code,
            level_2_code,
            taxonomy_version
        from tmm_mieventm
        where del_yn = 'N'
        order by first_created desc, mi_event_id desc
        limit :limit
    """)
    with engine.connect() as conn:
        rows = conn.execute(sql, {"limit": limit}).mappings().all()
    return pd.DataFrame(rows)

try:
    latest_events_df = fetch_latest_events(limit=10)
    latest_events_df
except Exception as exc:
    print("DB 최신 event 조회를 건너뜁니다:", exc)


## 11. 실행 결과 요약

이 데모에서 확인할 수 있는 핵심은 다음과 같습니다.

- `event_flow_agent`는 `/invoke` API를 통해 end-to-end로 실행된다.
- 원천 record를 조회하고, LLM을 통해 record 분석 및 event 추출을 수행한다.
- 추출된 event는 `tmm_mieventm` 테이블에 저장된다.
- 저장된 event를 다시 조회하여 최종 요약을 생성한다.
- 현재 구조는 PoC 및 후속 기능 확장을 위해 단계별 결과를 확인할 수 있도록 구성되어 있다.
